In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/legal-doc-classification/sample_submission.csv
/kaggle/input/legal-doc-classification/train.csv
/kaggle/input/legal-doc-classification/test.csv


In [2]:
!pip install -q -U accelerate evaluate

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import os, re, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback, set_seed
)

SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
Device: cuda


In [3]:
class CFG:
    # ---- workflow control ----
    debug_mode = False          # <-- START HERE. True = fast subset run to validate pipeline.
    debug_train_size = 600
    debug_val_size = 150
    debug_test_size = 100
    debug_epochs = 1
    full_epochs = 3

    train_path = "/kaggle/input/legal-doc-classification/train.csv"
    test_path  = "/kaggle/input/legal-doc-classification/test.csv"
    id_col = "id"; text_col = "text"; label_col = "label"

    model_name = "answerdotai/ModernBERT-base"
    max_length = 512
    truncation_strategy = "head_tail"
    head_len = 128; tail_len = 384
    preslice_head_words = 400; preslice_tail_words = 1200

    val_size = 0.15
    min_samples_per_class_for_stratify = 2
    lr = 2e-5
    train_bs = 4
    eval_bs = 16
    grad_accum = 4
    warmup_ratio = 0.06
    weight_decay = 0.01
    fp16 = torch.cuda.is_available()
    output_dir = "./modernbert_fullft"

    use_class_weights = True
    max_class_weight_ratio = 20.0

CFG.epochs = CFG.debug_epochs if CFG.debug_mode else CFG.full_epochs
print("Mode:", "DEBUG (subset)" if CFG.debug_mode else "FULL")
print("Epochs this run:", CFG.epochs)

Mode: FULL
Epochs this run: 3


In [4]:
train_df_full = pd.read_csv(CFG.train_path)
test_df_full  = pd.read_csv(CFG.test_path)
train_df_full = train_df_full.dropna(subset=[CFG.text_col, CFG.label_col]).reset_index(drop=True)
train_df_full[CFG.label_col] = train_df_full[CFG.label_col].astype(int)
CFG.num_labels = int(train_df_full[CFG.label_col].max()) + 1
print("num_labels:", CFG.num_labels)

def clean_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'Page\s*\d+\s*of\s*\d+', '', text, flags=re.I)
    text = re.sub(r'-{3,}', '', text)
    return text.strip()

def preslice_words(text, head_words, tail_words):
    words = text.split()
    if len(words) <= head_words + tail_words:
        return text
    return " ".join(words[:head_words] + words[-tail_words:])

def preprocess(df):
    df = df.copy()
    df[CFG.text_col] = df[CFG.text_col].apply(clean_text)
    df[CFG.text_col] = df[CFG.text_col].apply(
        lambda t: preslice_words(t, CFG.preslice_head_words, CFG.preslice_tail_words)
    )
    return df

train_df_full = preprocess(train_df_full)
test_df_full  = preprocess(test_df_full)
train_df, test_df = train_df_full, test_df_full

num_labels: 30


In [5]:
def safe_stratified_split(df, label_col, val_size, min_per_class, seed=SEED):
    counts = df[label_col].value_counts()
    splittable = counts[counts >= max(2, min_per_class)].index
    unsplittable = counts[counts < max(2, min_per_class)].index
    df_s = df[df[label_col].isin(splittable)]
    df_u = df[df[label_col].isin(unsplittable)]
    tr, va = train_test_split(df_s, test_size=val_size, stratify=df_s[label_col], random_state=seed)
    tr = pd.concat([tr, df_u]).sample(frac=1, random_state=seed).reset_index(drop=True)
    va = va.reset_index(drop=True)
    return tr, va

train_split, val_split = safe_stratified_split(
    train_df, CFG.label_col, CFG.val_size, CFG.min_samples_per_class_for_stratify
)
print("Train:", train_split.shape, "Val:", val_split.shape)

Train: (43214, 3) Val: (7626, 3)


In [6]:
def make_debug_subset(df, label_col, target_size, seed=SEED):
    """Stratified-ish sample: at least 1 per class, proportional fill up to target_size."""
    df = df.copy()
    parts = []
    for lbl, g in df.groupby(label_col):
        n_take = max(1, int(round(target_size * len(g) / len(df))))
        n_take = min(n_take, len(g))
        parts.append(g.sample(n=n_take, random_state=seed))
    out = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

if CFG.debug_mode:
    train_df = make_debug_subset(train_df_full, CFG.label_col, CFG.debug_train_size)
    test_df = test_df_full.sample(
        n=min(CFG.debug_test_size, len(test_df_full)), random_state=SEED
    ).reset_index(drop=True)
    print("DEBUG subset -> train:", train_df.shape, "test:", test_df.shape)
    print("Classes represented:", train_df[CFG.label_col].nunique(), "/", train_df_full[CFG.label_col].nunique())
else:
    train_df = train_df_full
    test_df = test_df_full
    print("FULL run -> train:", train_df.shape, "test:", test_df.shape)

FULL run -> train: (50840, 3) test: (12710, 2)


In [7]:
CFG.output_dir = "./modernbert_debug" if CFG.debug_mode else "./modernbert_fullft"

In [8]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

def head_tail_truncate(text, tokenizer, max_length, head_len, tail_len):
    ids = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
    max_body = max_length - 2
    if len(ids) <= max_body:
        body = ids
    else:
        h = min(head_len, max_body)
        t = max_body - h
        body = ids[:h] + ids[-t:]
    input_ids = [tokenizer.cls_token_id] + body + [tokenizer.sep_token_id]
    attn = [1] * len(input_ids)
    pad_len = max_length - len(input_ids)
    if pad_len > 0:
        input_ids += [tokenizer.pad_token_id] * pad_len
        attn += [0] * pad_len
    return input_ids[:max_length], attn[:max_length]

class DocDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts.tolist()
        self.labels = labels.tolist() if labels is not None else None
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        input_ids, attn = head_tail_truncate(self.texts[idx], tokenizer, CFG.max_length, CFG.head_len, CFG.tail_len)
        item = {"input_ids": torch.tensor(input_ids), "attention_mask": torch.tensor(attn)}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

train_ds = DocDataset(train_split[CFG.text_col], train_split[CFG.label_col])
val_ds   = DocDataset(val_split[CFG.text_col], val_split[CFG.label_col])
test_ds  = DocDataset(test_df[CFG.text_col])

model = AutoModelForSequenceClassification.from_pretrained(
    CFG.model_name, num_labels=CFG.num_labels
).to(device)
model.gradient_checkpointing_enable()   # saves memory since we're doing full FT, not LoRA

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
if CFG.use_class_weights:
    counts_in_train = train_split[CFG.label_col].value_counts().reindex(range(CFG.num_labels), fill_value=0)
    safe_counts = counts_in_train.replace(0, counts_in_train[counts_in_train > 0].min() if (counts_in_train > 0).any() else 1)
    raw_weights = (1.0 / safe_counts)
    raw_weights = raw_weights / raw_weights.mean()
    capped_weights = raw_weights.clip(lower=raw_weights.min(), upper=raw_weights.min() * CFG.max_class_weight_ratio)
    class_weights = torch.tensor(capped_weights.values, dtype=torch.float).to(device)
    print("Weight ratio:", (class_weights.max() / class_weights.min()).item())
else:
    class_weights = None

Weight ratio: 20.000001907348633


In [10]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits").float()
        loss_fct = nn.CrossEntropyLoss(weight=class_weights) if class_weights is not None else nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, CFG.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=CFG.output_dir,
    learning_rate=CFG.lr,
    per_device_train_batch_size=CFG.train_bs,
    per_device_eval_batch_size=CFG.eval_bs,
    gradient_accumulation_steps=CFG.grad_accum,
    num_train_epochs=CFG.epochs,
    weight_decay=CFG.weight_decay,
    warmup_ratio=CFG.warmup_ratio,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=CFG.fp16,
    logging_steps=10,
    report_to="none",
    seed=SEED,
    save_total_limit=2,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],   # stops early if val stops improving for 2 epochs
)

train_result = trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
W0712 07:30:13.581000 23 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,4.727640,1.262148,0.672174,0.490989
2,2.998006,1.122067,0.708497,0.539145
3,2.909767,1.204855,0.718725,0.560026


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
val_metrics = trainer.evaluate()
print(val_metrics)

val_logits = trainer.predict(val_ds).predictions
test_logits = trainer.predict(test_ds).predictions

import torch.nn.functional as F
val_probs_modernbert = F.softmax(torch.tensor(val_logits), dim=-1).numpy()
test_probs_modernbert = F.softmax(torch.tensor(test_logits), dim=-1).numpy()

np.save("val_probs_modernbert.npy", val_probs_modernbert)
np.save("test_probs_modernbert.npy", test_probs_modernbert)
np.save("val_labels.npy", val_split[CFG.label_col].values)

trainer.save_model("./modernbert_fullft_model")
tokenizer.save_pretrained("./modernbert_fullft_model")
print("Saved model + probabilities.")

{'eval_loss': 1.204854965209961, 'eval_accuracy': 0.7187254130605822, 'eval_f1_macro': 0.5600264213255666, 'eval_runtime': 124.8312, 'eval_samples_per_second': 61.09, 'eval_steps_per_second': 3.821, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model + probabilities.


In [12]:
test_preds_modernbert = np.argmax(test_probs_modernbert, axis=1)

submission_modernbert = pd.DataFrame({
    'ID': test_df['id'],
    CFG.label_col: test_preds_modernbert,
})

submission_modernbert.to_csv("submission.csv", index=False)
print("Saved submission.csv (solo ModernBERT full fine-tune)")
submission_modernbert.head()

Saved submission.csv (solo ModernBERT full fine-tune)


,ID,label
0,0,7
1,1,26
2,2,3
3,3,11
4,4,0


In [13]:
if CFG.debug_mode:
    print("""
    ============================================================
    DEBUG PASS COMPLETE.
    Check above for:
      - No errors/OOM during training or inference
      - eval_accuracy and eval_f1_macro are non-zero and reasonable
        (won't be good with 600 rows/1 epoch, just needs to be non-garbage)
      - Confusion matrix / classification_report ran without crashing

    If all clear:
      1. Scroll up to CFG and set debug_mode = False
      2. Restart the kernel (Run All from top, or just re-run all cells)
      3. This will train on the FULL dataset for CFG.full_epochs epochs
         with early stopping, and produce the real submission.csv
    ============================================================
    """)
else:
    print("FULL run complete. submission.csv is the real ModernBERT submission.")

FULL run complete. submission.csv is the real ModernBERT submission.
